# QUERY 5
Cantidad de transacciones del período [2022-09-01, 2022-09-05] con formato de pago
"Wire" o "ACH" cuyo monto convertido a USD sea menor a 1

In [33]:
import numpy as np
import pandas as pd
import requests

RUTA_DE_DATASETS = "../../datasets"
NOMBRE_ARCHIVO = "transacciones_sample.csv"

transacciones = pd.read_csv(f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO}")
print(f"Total filas cargadas: {len(transacciones)}")
transacciones.head()


Total filas cargadas: 10000


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/08 00:41,15783,804306930,1812,8129EE4B0,159.68,US Dollar,159.68,US Dollar,Cheque,0
1,2022/09/07 16:57,115,8000520B0,163897,818C52C60,1800564.28,Saudi Riyal,1800564.28,Saudi Riyal,ACH,0
2,2022/09/01 00:04,119732,80ECA8BE0,119732,80ECA8BE0,357098.69,Rupee,357098.69,Rupee,Reinvestment,0
3,2022/09/05 03:59,2557,801B34500,13590,801B32D60,42396.36,Euro,42396.36,Euro,ACH,0
4,2022/09/06 22:12,3460,801AB0F50,219760,8196C7940,109.42,Euro,109.42,Euro,Credit Card,0


In [ ]:
# Filtro por período [2022-09-01, 2022-09-05]
# "2022/09/01" <= Timestamp <= "2022/09/05"
INICIO = "2022/09/01"
FIN    = "2022/09/05"

filtro_periodo = transacciones[
    (transacciones["Timestamp"] >= INICIO) &
    (transacciones["Timestamp"] <= FIN)
]
print(f"Transacciones en el período: {len(filtro_periodo)}")
filtro_periodo.head()


Transacciones en el período: 4612


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
2,2022/09/01 00:04,119732,80ECA8BE0,119732,80ECA8BE0,357098.69,Rupee,357098.69,Rupee,Reinvestment,0
8,2022/09/01 20:11,21246,810E6F070,21246,810E6F070,959.33,US Dollar,959.33,US Dollar,Reinvestment,0
10,2022/09/02 00:00,359507,815B3FDA0,257858,815B3FD00,2296.74,Shekel,2296.74,Shekel,Cheque,0
11,2022/09/01 04:43,12204,8074C6F90,122996,809CBB3C0,473.56,US Dollar,473.56,US Dollar,Credit Card,0
14,2022/09/01 00:03,158053,815F8DEF0,158053,815F8DEF0,33525.54,Shekel,33525.54,Shekel,Reinvestment,0


In [35]:
# Filtro por formato de pago: Wire o ACH
FORMATOS = ["Wire", "ACH"]

filtro_formato = filtro_periodo[filtro_periodo["Payment Format"].isin(FORMATOS)]
print(f"Transacciones Wire/ACH en el período: {len(filtro_formato)}")
filtro_formato.head()


Transacciones Wire/ACH en el período: 633


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
17,2022/09/01 00:08,115481,8169CCA20,240478,8169CC900,2945.00,US Dollar,2945.00,US Dollar,ACH,0
27,2022/09/02 03:47,325325,80CE0F890,215107,80CE10010,493.05,US Dollar,493.05,US Dollar,ACH,0
33,2022/09/01 00:22,343628,817255D50,224231,817243B60,4954.14,US Dollar,4954.14,US Dollar,ACH,0
39,2022/09/04 21:40,22435,80192DD50,22124,801FEE310,104583.08,US Dollar,104583.08,US Dollar,ACH,0
42,2022/09/02 09:38,21260,80093B810,21260,80093B810,262886.28,Swiss Franc,287307.41,US Dollar,ACH,0


In [36]:
# Descarga de cotizaciones desde API Frankfurter (base EUR)
# y forward-fill para días sin mercado (fines de semana / feriados)
from datetime import date, timedelta

url = "https://api.frankfurter.app/2022-09-01..2022-09-05"
resp = requests.get(url, timeout=10)
resp.raise_for_status()
cotizaciones_raw = resp.json()["rates"]   # solo días hábiles
print("Fechas con cotización de la API:", list(cotizaciones_raw.keys()))

# Expandir el rango completo usando la última cotización conocida
cotizaciones = {}
last_rates = None
dia = date(2022, 9, 1)
fin = date(2022, 9, 5)
while dia <= fin:
    key = dia.strftime("%Y-%m-%d")
    if key in cotizaciones_raw:
        last_rates = cotizaciones_raw[key]
    if last_rates is not None:
        cotizaciones[key] = last_rates
    dia += timedelta(days=1)

print("Fechas disponibles tras forward-fill:", list(cotizaciones.keys()))


Fechas con cotización de la API: ['2022-09-01', '2022-09-02', '2022-09-05']
Fechas disponibles tras forward-fill: ['2022-09-01', '2022-09-02', '2022-09-03', '2022-09-04', '2022-09-05']


In [37]:
# Conversión de monto recibido a USD (misma lógica que el worker distribuido)
CURRENCY_MAP = {
    "US Dollar":         "USD",
    "Euro":              "EUR",
    "UK Pound":          "GBP",
    "Yen":               "JPY",
    "Australian Dollar": "AUD",
    "Bitcoin":           "BTC",
    "Brazil Real":       "BRL",
    "Canadian Dollar":   "CAD",
    "Mexican Peso":      "MXN",
    "Ruble":             "RUB",
    "Rupee":             "INR",
    "Saudi Riyal":       "SAR",
    "Shekel":            "ILS",
    "Swiss Franc":       "CHF",
    "Yuan":              "CNY",
}

def convertir_a_usd(row):
    iso = CURRENCY_MAP.get(row["Receiving Currency"])
    if not iso:
        return None
    if iso == "USD":
        return float(row["Amount Received"])

    fecha = row["Timestamp"].split(" ")[0].replace("/", "-")
    rates = cotizaciones.get(fecha)  # ya tiene forward-fill para fines de semana
    if not rates:
        return None

    rate_usd = rates.get("USD")
    if not rate_usd:
        return None

    monto = float(row["Amount Received"])
    if iso == "EUR":
        return monto * rate_usd

    # Triangulación: origen → EUR → USD
    rate_origen = rates.get(iso)
    if not rate_origen:
        return None
    return (monto / rate_origen) * rate_usd

filtro_formato = filtro_formato.copy()
filtro_formato["amount_usd"] = filtro_formato.apply(convertir_a_usd, axis=1)

# Descartamos filas sin cotización (divisa no mapeada) y filtramos monto < 1 USD
menores_1_usd = filtro_formato.dropna(subset=["amount_usd"])
menores_1_usd = menores_1_usd[menores_1_usd["amount_usd"] < 1.0]
print(f"Transacciones < 1 USD: {len(menores_1_usd)}")
menores_1_usd[["Timestamp", "Receiving Currency", "Amount Received", "amount_usd"]].head()


Transacciones < 1 USD: 14


,Timestamp,Receiving Currency,Amount Received,amount_usd
1340,2022/09/01 00:08,Yen,0.18,0.001292
1660,2022/09/01 00:04,Euro,0.01,0.010004
2193,2022/09/01 03:48,Australian Dollar,0.07,0.047797
3160,2022/09/01 00:04,Rupee,0.13,0.001633
3522,2022/09/01 19:39,Euro,0.67,0.670268


In [38]:
# Resultado final: cantidad de transacciones
count = len(menores_1_usd)
resultado = pd.DataFrame({"count": [count]})
print(f"Q5 resultado: {count} transacciones")

resultado.to_csv("output_q5.csv", index=False)
print("Guardado en output_q5.csv")


Q5 resultado: 14 transacciones
Guardado en output_q5.csv
